# Advanced Breast Cancer Classification with Neural Networks

This notebook is an advanced version of the simple breast cancer neural-network project.

It adds:

- Exploratory Data Analysis
- Stratified train/validation/test split
- Scaling
- Baseline Logistic Regression
- Regularized Keras Neural Network
- Early stopping
- Accuracy, Precision, Recall, F1, ROC-AUC
- Confusion matrix, ROC curve, Precision-Recall curve
- Threshold tuning
- Model/scaler saving

**Disclaimer:** This is an educational project only and must not be used as a real medical diagnostic tool.

In [ ]:
# If TensorFlow is missing, install packages first in your selected environment:
# %pip install tensorflow pandas scikit-learn matplotlib joblib

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 1. Load the dataset

The target column has two classes:

- `0` = malignant
- `1` = benign

In [ ]:
dataset = load_breast_cancer()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df["label"] = dataset.target

df.head()

In [ ]:
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Class distribution:")
print(df["label"].value_counts())

## 2. Quick EDA

In [ ]:
class_counts = df["label"].value_counts().sort_index()
plt.figure(figsize=(5, 4))
plt.bar(["Malignant", "Benign"], class_counts.values)
plt.title("Target Distribution")
plt.ylabel("Count")
plt.show()

In [ ]:
# Show the features most correlated with the label
corr_with_target = df.corr(numeric_only=True)["label"].drop("label").sort_values(key=abs, ascending=False)
corr_with_target.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
top_corr = corr_with_target.head(10).sort_values()
plt.barh(top_corr.index, top_corr.values)
plt.title("Top 10 Features Correlated with Target")
plt.xlabel("Correlation with label")
plt.tight_layout()
plt.show()

## 3. Train / validation / test split

We use **stratified splitting** so the malignant/benign ratio stays similar in each split.

In [ ]:
X = df.drop(columns="label")
y = df["label"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15 / 0.85,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

## 4. Scaling

Neural networks train better when numeric features are standardized.
The scaler is fitted only on the training data to avoid data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 5. Baseline model: Logistic Regression

A good portfolio project should compare the neural network with a simpler baseline.

In [ ]:
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_scaled, y_train)

baseline_prob = baseline.predict_proba(X_test_scaled)[:, 1]
baseline_pred = (baseline_prob >= 0.5).astype(int)

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred),
    "recall": recall_score(y_test, baseline_pred),
    "f1": f1_score(y_test, baseline_pred),
    "roc_auc": roc_auc_score(y_test, baseline_prob),
}

baseline_metrics

## 6. Advanced Neural Network

This version uses:

- Dense layers
- Batch Normalization
- Dropout
- L2 regularization
- Binary output with `sigmoid`
- `binary_crossentropy`

In [ ]:
def build_model(input_dim: int, learning_rate: float = 0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.30),

        layers.Dense(32, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.20),

        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model

model = build_model(X_train_scaled.shape[1])
model.summary()

## 7. Train with callbacks

Early stopping prevents unnecessary training and reduces overfitting.

In [ ]:
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
class_weights = {i: weight for i, weight in enumerate(class_weights_array)}
class_weights

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=20,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=8,
        min_lr=1e-5,
    ),
]

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)

## 8. Training curves

In [ ]:
def plot_history(history, metric):
    plt.figure(figsize=(7, 4))
    plt.plot(history.history[metric], label=f"train_{metric}")
    plt.plot(history.history[f"val_{metric}"], label=f"val_{metric}")
    plt.title(f"Training vs Validation {metric}")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_history(history, "loss")
plot_history(history, "accuracy")
plot_history(history, "auc")

## 9. Threshold tuning

The default threshold is `0.5`, but we can tune it on the validation set to optimize F1-score.

In [ ]:
def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_threshold = 0.5
    best_f1 = -1

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

val_prob = model.predict(X_val_scaled).ravel()
best_threshold, best_val_f1 = find_best_threshold(y_val, val_prob)
best_threshold, best_val_f1

## 10. Final evaluation on test data

In [ ]:
test_prob = model.predict(X_test_scaled).ravel()
test_pred = (test_prob >= best_threshold).astype(int)

nn_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred),
    "recall": recall_score(y_test, test_pred),
    "f1": f1_score(y_test, test_pred),
    "roc_auc": roc_auc_score(y_test, test_prob),
}

print("Baseline Logistic Regression:")
print(baseline_metrics)
print("
Neural Network:")
print(nn_metrics)
print("
Classification Report:")
print(classification_report(y_test, test_pred, target_names=["malignant", "benign"]))

In [ ]:
cm = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"]).plot(cmap="Blues", values_format="d")
plt.title("Neural Network Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, test_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, test_prob)
plt.figure(figsize=(6, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

## 11. Prediction system

In [ ]:
example_input = np.array([
    11.76, 21.6, 74.72, 427.9, 0.08637, 0.04966, 0.01657, 0.01115,
    0.1495, 0.05888, 0.4062, 1.21, 2.635, 28.47, 0.005857, 0.009758,
    0.01168, 0.007445, 0.02406, 0.001769, 12.98, 25.72, 82.98, 516.5,
    0.1085, 0.08615, 0.05523, 0.03715, 0.2433, 0.06563,
]).reshape(1, -1)

example_scaled = scaler.transform(example_input)
probability_benign = model.predict(example_scaled).ravel()[0]
prediction = int(probability_benign >= best_threshold)

print("Probability benign:", probability_benign)
print("Prediction:", "Benign" if prediction == 1 else "Malignant")

## 12. Save model, scaler, feature names, and metadata

In [ ]:
model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

model.save(model_dir / "breast_cancer_nn.keras")
joblib.dump(scaler, model_dir / "scaler.joblib")
joblib.dump(list(X.columns), model_dir / "feature_names.joblib")

metadata = {
    "target_mapping": {"0": "malignant", "1": "benign"},
    "best_threshold": float(best_threshold),
    "baseline_logistic_regression": baseline_metrics,
    "neural_network": nn_metrics,
}

with open(model_dir / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

metadata

## GitHub summary

This project is now stronger because it does not only train a neural network. It also shows a full ML workflow: data preparation, baseline comparison, advanced model, validation strategy, serious metrics, threshold tuning, saving artifacts, and deployment-ready prediction logic.